In [ ]:
# ============================================================
# CELL 0: Installs & Imports
# ============================================================
# Run this cell first. Colab may need to restart after install.

!pip install geemap scikit-learn --quiet

import ee
import geemap
import numpy as np
import pandas as pd
import math
import json
import datetime
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

try:
    ee.Initialize(project='replicating-paper')
    print("Google Earth Engine Initialized successfully.")
except Exception as e:
    print("Authentication required...")
    ee.Authenticate()
    ee.Initialize(project='replicating-paper')
    print("Google Earth Engine Authenticated and Initialized.")

print("All imports loaded.")

Authentication required...
Google Earth Engine Authenticated and Initialized.
All imports loaded.


In [ ]:
# ============================================================
# CELL 1: Pipeline Config
# This is the ONLY cell you edit when switching ROIs.
# All parameters that used to be hard-coded are now here.
# ============================================================

CONFIG = {
    # --- Identity ---
    "roi_name":      "Sanjay Van, Delhi",
    "project_id":    "replicating-paper",

    # --- ROI (rectangle for now; replace with polygon coords for real ROIs) ---
    # [west, south, east, north]
    "roi_bounds":    [77.16, 28.51, 77.19, 28.55],

    # --- Time windows ---
    # Phenology (HLS): 10-year window. Update END_DATE each year.
    "pheno_start":   "2013-01-01",
    "pheno_end":     "2024-12-31",

    # Radar (S1): same long window
    "radar_start":   "2014-04-01",   # S1A launched April 2014
    "radar_end":     "2024-12-31",

    # S2 composites for SNIC: single recent year, 4 seasons
    # Format: [("season_name", start_MMDD, end_MMDD)]
    "s2_year":       2023,
    "s2_seasons":    [
        ("winter",      "11-01", "01-31"),
        ("pre_monsoon", "03-01", "05-31"),
        ("monsoon",     "06-15", "09-15"),
        ("post_monsoon","09-16", "10-31"),
    ],

    # --- Resolution ---
    # 10m for ROIs ≤ 1000 km²; increase for larger ROIs.
    "analysis_scale": 10,

    # --- Masking thresholds ---
    "cloud_pct_max": 20,             # Max scene cloud % for S2/HLS pre-filter
    # VIIRS: we use the 95th percentile within the buffered ROI (auto-set in Cell 3)
    # so no hard-coded value here. The old hardcoded 30 is gone.

    # --- SNIC ---
    # seed_spacing auto-scales to target ~5000 stands (set in Cell 9).
    "snic_compactness":    0.4,
    "snic_connectivity":   8,
    "snic_neighborhood":   128,
    "snic_target_stands":  5000,

    # --- Phenology (HLS dual harmonic) ---
    "n_curve_samples":     24,       # Points sampled from fitted curve per year
    "n_pca_components":    5,        # PCA reduction of 24 phenology features

    # --- Clustering ---
    "k_range":       [2, 15],        # Range tested for auto-K selection
    "k_sample_pts":  1500,           # Points sampled for silhouette/DB scoring

    # --- Export ---
    "export_asset_base": "projects/replicating-paper/assets/",
}

# ---- Derived geometry (don't edit) ----
CONFIG["roi"] = ee.Geometry.Rectangle(CONFIG["roi_bounds"])

# Buffer for VIIRS threshold calculation (5 km around ROI)
_bounds = CONFIG["roi_bounds"]
CONFIG["roi_buffered"] = ee.Geometry.Rectangle([
    _bounds[0] - 0.05, _bounds[1] - 0.05,
    _bounds[2] + 0.05, _bounds[3] + 0.05,
])

print(f"Config loaded for ROI: {CONFIG['roi_name']}")
print(f"  Phenology window : {CONFIG['pheno_start']} → {CONFIG['pheno_end']}")
print(f"  Analysis scale   : {CONFIG['analysis_scale']} m")
print(f"  SNIC compactness : {CONFIG['snic_compactness']}")
print(f"  Curve samples    : {CONFIG['n_curve_samples']}  →  PCA to {CONFIG['n_pca_components']} components")

Config loaded for ROI: Sanjay Van, Delhi
  Phenology window : 2013-01-01 → 2024-12-31
  Analysis scale   : 10 m
  SNIC compactness : 0.4
  Curve samples    : 24  →  PCA to 5 components


In [ ]:
# ============================================================
# CELL 3: Module 1 — Pre-processing & Masking
#
# CHANGES FROM v1:
#   - VIIRS threshold is now percentile-based (no longer
#     the Delhi-calibrated hard-coded 30).
#   - Kept WorldCover + JRC Water as before.
# ============================================================

roi     = CONFIG["roi"]
roi_buf = CONFIG["roi_buffered"]
scale   = CONFIG["analysis_scale"]

# ── 1. ESA WorldCover v200 ─────────────────────────────────
worldcover  = ee.ImageCollection("ESA/WorldCover/v200").filterBounds(roi).first()
lc_map      = worldcover.select('Map')
is_veg      = lc_map.eq(10).Or(lc_map.eq(20)).Or(lc_map.eq(30))
print("WorldCover: Trees(10), Shrubland(20), Grassland(30).")

# ── 2. JRC Global Surface Water ───────────────────────────
water       = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select('occurrence').clip(roi).unmask(0)
is_dry      = water.eq(0)
print("JRC Surface Water mask ready.")

# ── 3. VIIRS Night Lights — percentile-based threshold ────
# v1 hard-coded 30 (calibrated to Delhi). Here we compute the
# 95th percentile of VIIRS within a 5 km buffer of the ROI.
# This auto-adapts: a dense city gets a high threshold; a
# pristine forest gets a low one. Floor at 1.5 to avoid
# accidentally excluding everything in a dark landscape.

viirs_mean = (
    ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG")
    .filterBounds(roi_buf)
    .filterDate(CONFIG["pheno_start"], CONFIG["pheno_end"])
    .select("avg_rad")
    .mean()
    .clip(roi_buf)
)

viirs_stats = viirs_mean.reduceRegion(
    reducer   = ee.Reducer.percentile([95]),
    geometry  = roi_buf,
    scale     = 1000,
    maxPixels = 1e9,
).getInfo()

viirs_threshold_val = list(viirs_stats.values())[0] if viirs_stats else 30.0
viirs_threshold = ee.Number(max(float(viirs_threshold_val), 1.5))

is_dark = viirs_mean.clip(roi).lt(viirs_threshold)

print(f"VIIRS threshold (p95 of buffered ROI): {viirs_threshold.getInfo():.2f} nW/cm²/sr")
print("Pixels above this are flagged as urban/lit and excluded.")

# ── 4. Combine into suitability mask ──────────────────────
valid_mask = is_veg.And(is_dry).And(is_dark).rename("Suitability_Mask")

# ── 5. Report survival rate ────────────────────────────────
total_px = ee.Image.constant(1).reduceRegion(
    reducer=ee.Reducer.count(), geometry=roi, scale=scale, maxPixels=1e9
).get("constant").getInfo()

valid_px = valid_mask.reduceRegion(
    reducer=ee.Reducer.sum(), geometry=roi, scale=scale, maxPixels=1e9
).get("Suitability_Mask").getInfo()

print(f"\n--- Masking Report ---")
print(f"Total ROI pixels ({scale}m): {int(total_px)}")
print(f"Valid forest pixels        : {int(valid_px)}")
print(f"Survival rate              : {100 * valid_px / total_px:.1f}%")

# ── 6. Quick visualization ────────────────────────────────
Map_m1 = geemap.Map()
Map_m1.add_basemap("HYBRID")
Map_m1.centerObject(roi, 14)
Map_m1.addLayer(valid_mask, {"min": 0, "max": 1, "palette": ["black", "00FF00"]},
                "Module 1: Valid mask (green = forest)")
Map_m1

WorldCover: Trees(10), Shrubland(20), Grassland(30).
JRC Surface Water mask ready.
VIIRS threshold (p95 of buffered ROI): 58.59 nW/cm²/sr
Pixels above this are flagged as urban/lit and excluded.

--- Masking Report ---
Total ROI pixels (10m): 148630
Valid forest pixels        : 103128
Survival rate              : 69.4%


Map(center=[28.529999558822738, 77.17500000000798], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
# ============================================================
# CELL 4: Module 2 — HLS Phenology
#
# WHAT CHANGED FROM v1:
#   - Data source: S2_SR_HARMONIZED → HLS (L30 + S30 merged).
#     HLS gives 10+ years and 2-4× more cloud-free obs/year.
#   - Vegetation index: NDVI → NIRv (NIRv = NDVI × NIR).
#     NIRv avoids saturation in dense canopy and tracks GPP
#     more linearly.
#   - Harmonic model: single annual → DUAL harmonic (annual
#     + semi-annual). The 4π terms are ESSENTIAL for bimodal
#     Indian monsoon + post-monsoon phenology.
# ============================================================

roi          = CONFIG["roi"]
pheno_start  = CONFIG["pheno_start"]
pheno_end    = CONFIG["pheno_end"]
valid_mask   = valid_mask   # from Cell 3

# ── Helper: HLS Fmask cloud masking ───────────────────────
# HLS Fmask v002 bit layout:
#   Bit 0: Cirrus | Bit 1: Cloud shadow | Bit 2: Adjacent cloud
#   Bit 3: Snow/ice | Bit 4: Water
#   Bit 5: Cloud high confidence | Bit 6: Cloud medium conf.
# We keep pixels where bits 1 (shadow) and 5 (cloud) are 0.

def mask_hls_clouds(img):
    fmask   = img.select("Fmask")
    no_cloud  = fmask.bitwiseAnd(1 << 5).eq(0)  # bit 5 = cloud
    no_shadow = fmask.bitwiseAnd(1 << 1).eq(0)  # bit 1 = shadow
    return img.updateMask(no_cloud.And(no_shadow))

# ── Helper: rename S30 bands to match L30 naming ──────────
# L30: B4=Red, B5=NIR
# S30: B4=Red, B8A=NIR (narrow) → rename to B5 for merge
def rename_s30(img):
    # Select only what add_nirv_and_time needs: Red (B4) and NIR (B8A→B5)
    # Avoids clash with the existing S30 B5 (which is a different SWIR band)
    return img.select(['B4', 'B8A']).rename(['B4', 'B5'])

# ── Helper: compute NIRv and add time bands ────────────────
def add_nirv_and_time(img):
    nir  = img.select("B5")
    red  = img.select("B4")
    ndvi = nir.subtract(red).divide(nir.add(red))
    nirv = ndvi.multiply(nir).rename("NIRv")

    # t = years elapsed since pheno_start (continuous)
    t = img.date().difference(ee.Date(pheno_start), "year")
    t_img = ee.Image.constant(t).toFloat()

    # Dual harmonic regressors: annual (2π) + semi-annual (4π)
    two_pi_t  = t_img.multiply(2 * math.pi)
    four_pi_t = t_img.multiply(4 * math.pi)

    return img.addBands([
        nirv,
        t_img.rename("t"),
        ee.Image.constant(1).rename("constant"),
        t_img.rename("trend"),              # β₁ (linear trend)
        two_pi_t.cos().rename("cos1"),      # β₂ (annual cosine)
        two_pi_t.sin().rename("sin1"),      # β₃ (annual sine)
        four_pi_t.cos().rename("cos2"),     # β₄ (semi-annual cosine)
        four_pi_t.sin().rename("sin2"),     # β₅ (semi-annual sine)
    ])

# ── Load & merge HLS L30 + S30 ────────────────────────────
print("Loading HLS L30 (Landsat) ...")
hls_l30 = (
    ee.ImageCollection("NASA/HLS/HLSL30/v002")
    .filterBounds(roi)
    .filterDate(pheno_start, pheno_end)
    .map(mask_hls_clouds)
    .map(add_nirv_and_time)
)

print("Loading HLS S30 (Sentinel-2) ...")
hls_s30 = (
    ee.ImageCollection("NASA/HLS/HLSS30/v002")
    .filterBounds(roi)
    .filterDate(pheno_start, pheno_end)
    .map(mask_hls_clouds)
    .map(rename_s30)
    .map(add_nirv_and_time)
)

# Merge into one collection — this is what "HLS" means
hls_merged = hls_l30.merge(hls_s30)

n_obs = hls_merged.size().getInfo()
print(f"\nTotal HLS observations (cloud-masked, merged): {n_obs}")
print("(vs ~100-150 you'd get from S2 alone over 5 years)")

# ── Fit the dual harmonic regression ──────────────────────
# Regressors: [constant, trend, cos1, sin1, cos2, sin2]  (6 inputs, 1 output)
print("\nFitting dual harmonic regression on NIRv time series ...")

regressors = ["constant", "trend", "cos1", "sin1", "cos2", "sin2"]
regression = (
    hls_merged
    .select(regressors + ["NIRv"])
    .reduce(ee.Reducer.linearRegression(numX=6, numY=1))
)

# Unpack the 6 coefficients into named bands
coeffs = (
    regression.select("coefficients")
    .arrayProject([0])
    .arrayFlatten([["beta0", "beta1", "beta2", "beta3", "beta4", "beta5"]])
)

# Apply valid mask (30m resolution; will be upsampled later at per-stand stage)
hls_coeffs = coeffs.updateMask(valid_mask.reproject("EPSG:4326", None, 30))

print("Dual harmonic coefficients computed:")
print("  β0 = intercept (NIRv baseline)")
print("  β1 = linear trend (greening / browning over 10 years)")
print("  β2,β3 = annual cosine/sine amplitude and phase")
print("  β4,β5 = semi-annual cosine/sine (monsoon bimodality)")
print("\nDone. hls_coeffs ready (6 bands, 30m).")

Loading HLS L30 (Landsat) ...
Loading HLS S30 (Sentinel-2) ...

Total HLS observations (cloud-masked, merged): 10786
(vs ~100-150 you'd get from S2 alone over 5 years)

Fitting dual harmonic regression on NIRv time series ...
Dual harmonic coefficients computed:
  β0 = intercept (NIRv baseline)
  β1 = linear trend (greening / browning over 10 years)
  β2,β3 = annual cosine/sine amplitude and phase
  β4,β5 = semi-annual cosine/sine (monsoon bimodality)

Done. hls_coeffs ready (6 bands, 30m).


In [ ]:
# ── CELL 4b: Quick sanity check — click a pixel, see NIRv + dual harmonic fit ──

Map_check = geemap.Map()
Map_check.centerObject(roi, 15)
Map_check.addLayer(valid_mask, {"min": 0, "max": 1, "palette": ["black", "00FF00"]},
                   "Valid forest (click here)")
out = widgets.Output(layout={"border": "1px solid #888", "padding": "8px"})
# Add this before the map/click handler, outside the click function
# Replace the hls_lean line with this fresh lightweight collection
def add_nirv_only(img):
    nir  = img.select("B5")
    red  = img.select("B4")
    ndvi = nir.subtract(red).divide(nir.add(red))
    nirv = ndvi.multiply(nir).rename("NIRv")
    t    = img.date().difference(ee.Date(pheno_start), "year")
    t_img = ee.Image.constant(t).toFloat().rename("t")
    return ee.Image.cat([t_img, nirv])

hls_lean_l30 = (ee.ImageCollection("NASA/HLS/HLSL30/v002")
    .filterBounds(roi).filterDate("2016-01-01", pheno_end)
    .map(mask_hls_clouds)
    .map(lambda img: img.select(["B4", "B5"]))
    .map(add_nirv_only))

hls_lean_s30 = (ee.ImageCollection("NASA/HLS/HLSS30/v002")
    .filterBounds(roi).filterDate("2016-01-01", pheno_end)
    .map(mask_hls_clouds)
    .map(rename_s30)
    .map(add_nirv_only))

hls_lean = hls_lean_l30.merge(hls_lean_s30)
def handle_check(**kwargs):
    if kwargs.get("type") != "click":
        return
    latlon = kwargs.get("coordinates")
    with out:
        out.clear_output(wait=True)
        print("Fetching...")
        try:
            point = ee.Geometry.Point(latlon[::-1])

            # Raw NIRv observations
            ts = hls_lean.getRegion(point, 30).getInfo()
            header = ts[0]
            t_idx, nirv_idx = header.index("t"), header.index("NIRv")
            raw_t, raw_nirv = [], []
            for row in ts[1:]:
                if row[t_idx] is not None and row[nirv_idx] is not None:
                    raw_t.append(row[t_idx])
                    raw_nirv.append(row[nirv_idx])

            # Fitted coefficients for this pixel
            p = hls_coeffs.reduceRegion(ee.Reducer.first(), point, 30).getInfo()
            b0, b1 = p.get("beta0", 0), p.get("beta1", 0)
            b2, b3 = p.get("beta2", 0), p.get("beta3", 0)
            b4, b5 = p.get("beta4", 0), p.get("beta5", 0)

            # Dual harmonic curve
            t_sm = np.linspace(0, 11, 500)
            curve = (b0 + b1*t_sm
                     + b2*np.cos(2*np.pi*t_sm) + b3*np.sin(2*np.pi*t_sm)
                     + b4*np.cos(4*np.pi*t_sm) + b5*np.sin(4*np.pi*t_sm))

            plt.figure(figsize=(10, 4))
            plt.scatter(raw_t, raw_nirv, color="darkgreen", alpha=0.5, s=20,
                        label=f"Raw HLS NIRv ({len(raw_t)} obs)")
            plt.plot(t_sm, curve, color="red", lw=2, label="Dual harmonic fit")
            plt.xlabel("Years from 2013"); plt.ylabel("NIRv")
            plt.ylim(-0.05, 0.7); plt.legend(); plt.grid(True, alpha=0.4)
            plt.title(f"β0={b0:.3f}  β1={b1:.4f}  |  Check bimodal shape in monsoon years")
            plt.show()
            print(f"{len(raw_t)} valid observations at this pixel.")
        except Exception as e:
            print(f"Error: {e}")

Map_check.on_interaction(handle_check)
display(widgets.VBox([Map_check, out]))
print("Click any green pixel to verify the dual harmonic fit before running PCA.")

Click any green pixel to verify the dual harmonic fit before running PCA.


In [ ]:
# ============================================================
# CELL 5: Module 2 (continued) — 24-Sample Curve Representation + PCA
# ============================================================

N        = CONFIG["n_curve_samples"]    # 24
N_PCA    = CONFIG["n_pca_components"]  # 5
scale    = 30  # HLS is 30m — phenology lives here until per-stand aggregation

# ── Step 1: Sample the fitted curve at N evenly-spaced points ─
# We evaluate the SEASONAL component only (no β1 trend),
# at fractional year t ∈ [0, 1).
# β1 is added as a SEPARATE feature (the 25th), so clusters
# distinguish both shape AND trajectory (design doc Option II).

sampled_bands = []
t_fractions = [k / N for k in range(N)]

for k, t_frac in enumerate(t_fractions):
    t_rad_annual     = 2 * math.pi * t_frac
    t_rad_semiannual = 4 * math.pi * t_frac

    # y(t) = β0 + β2·cos(2πt) + β3·sin(2πt) + β4·cos(4πt) + β5·sin(4πt)
    # (β1 trend excluded from the curve; added separately below)
    sample = (
        hls_coeffs.select("beta0")
        .add(hls_coeffs.select("beta2").multiply(math.cos(t_rad_annual)))
        .add(hls_coeffs.select("beta3").multiply(math.sin(t_rad_annual)))
        .add(hls_coeffs.select("beta4").multiply(math.cos(t_rad_semiannual)))
        .add(hls_coeffs.select("beta5").multiply(math.sin(t_rad_semiannual)))
        .rename(f"NIRv_t{k:02d}")
    )
    sampled_bands.append(sample)

# Stack the 24 bands into one image
curve_24 = ee.Image.cat(sampled_bands).updateMask(valid_mask.reproject("EPSG:4326", None, 30))
print(f"Sampled curve: {N} bands at 30m.")

# ── Step 2: PCA — reduce 24 bands → N_PCA principal components ─
print(f"Computing PCA: 24 bands → {N_PCA} components ...")

# 2a. Mean-center the 24-band image
band_names_24 = curve_24.bandNames()
means_dict = curve_24.reduceRegion(
    reducer   = ee.Reducer.mean(),
    geometry  = roi,
    scale     = 30,
    maxPixels = 1e9,
    bestEffort= True,
)
means_img   = ee.Image.constant(means_dict.values(band_names_24))
centered_24 = curve_24.subtract(means_img)

# 2b. Compute covariance matrix (24×24)
array_img = centered_24.toArray()
covar_dict = array_img.reduceRegion(
    reducer   = ee.Reducer.centeredCovariance(),
    geometry  = roi,
    scale     = 30,
    maxPixels = 1e9,
    bestEffort= True,
)
covar_arr = ee.Array(covar_dict.get("array"))

# 2c. Eigendecomposition — eigenvectors sorted by descending eigenvalue
eigens        = covar_arr.eigen()
# eigens[:,0]  = eigenvalues; eigens[:,1:] = eigenvectors
top_vectors   = eigens.slice(1, 1).slice(0, 0, N_PCA)  # shape: [N_PCA, 24]

# 2d. Pull eigenvectors to Python (5×24 = tiny, fine to fetch)
eigvec_list = top_vectors.toList().getInfo()  # list of 5 lists of 24 floats

# 2e. Project: each PC = weighted sum of the 24 centered bands
band_list = centered_24.bandNames().getInfo()
pc_images = []

for i in range(N_PCA):
    weights = eigvec_list[i]  # 24 floats for this PC
    # new — stack as multi-band image, then sum across bands
    weighted_img = ee.Image.cat([
        centered_24.select(band_list[j]).multiply(weights[j]).rename(f"w{j}")
        for j in range(N)
    ])
    pc_k = weighted_img.reduce(ee.Reducer.sum()).rename(f"Pheno_PC{i+1}")
    pc_images.append(pc_k)

pheno_pcs = ee.Image.cat(pc_images)

# ── Step 3: Add β1 trend as an explicit feature ──────────
pheno_trend = hls_coeffs.select("beta1").rename("Pheno_Trend")

# ── Final phenology feature image: N_PCA + 1 bands at 30m ─
pheno_features = ee.Image.cat([pheno_pcs, pheno_trend])

print(f"\nPhenology features ready: {N_PCA} PCs + 1 trend band = {N_PCA + 1} features at 30m.")
print("These will be upsampled to 10m at the per-stand aggregation stage (Module 9).")
print("They are NOT used in SNIC segmentation (by design — see §3.2 of design doc).")

Sampled curve: 24 bands at 30m.
Computing PCA: 24 bands → 5 components ...

Phenology features ready: 5 PCs + 1 trend band = 6 features at 30m.
These will be upsampled to 10m at the per-stand aggregation stage (Module 9).
They are NOT used in SNIC segmentation (by design — see §3.2 of design doc).


In [ ]:
print(f"HLS obs count: {n_obs}")
# Sanjay Van should be ~400–700 over 10 years (vs ~150 from S2 alone)
# If you see <100, the ROI bounds may not be intersecting the HLS tiles correctly

# Count actual valid (unmasked) pixels per image, then sum
def count_valid(img):
    valid = img.select("NIRv").mask().reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=roi,
        scale=30,
        maxPixels=1e9
    ).get("NIRv")
    return img.set("valid_px", valid)

valid_counts = hls_merged.map(count_valid)
actually_useful = valid_counts.filter(ee.Filter.gt("valid_px", 0))
print(f"Images with at least 1 valid pixel over ROI: {actually_useful.size().getInfo()}")

HLS obs count: 10786
Images with at least 1 valid pixel over ROI: 919


In [ ]:
# ============================================================
# CELL 6: Module 3 — Sentinel-1 Radar Features
#
# SAME APPROACH AS v1 (VH/VV ratio p10/p90/stdDev) but now
# using the full 10-year radar window and noting the S1B gap.
#
# NOTE on S1 timeline:
#   S1A: 3 Apr 2014 | S1B: 25 Apr 2016 – 3 Aug 2022 (anomaly)
#   S1C: operational May 2025 | S1D: launched Nov 2025
#   → The 2022–2024 period has lower revisit (12 days vs 6).
#   Percentile stats (p10/p90) are robust to this density
#   variation. We accept the limitation for v1.
# ============================================================

radar_start = CONFIG["radar_start"]
radar_end   = CONFIG["radar_end"]

s1_archive = (
    ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterBounds(roi)
    .filterDate(radar_start, radar_end)
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
    .filter(ee.Filter.eq("instrumentMode", "IW"))
)

n_s1 = s1_archive.size().getInfo()
print(f"S1 observations loaded: {n_s1} (window: {radar_start} → {radar_end})")

# ── VH/VV ratio in dB (VH − VV in dB space) ─────────────
# Ratio cancels incidence-angle effects that dominate raw VH.
def add_ratio(img):
    ratio = img.select("VH").subtract(img.select("VV")).rename("VH_VV_Ratio")
    return img.addBands(ratio)

s1_with_ratio = s1_archive.map(add_ratio)

# ── Three percentile-derived features ─────────────────────
# p90 = peak canopy/volume scattering (leaf-on)
# p10 = peak ground/stem scattering (leaf-off / sparse)
# stdDev = structural plasticity / seasonal dynamic range
s1_percentiles = (
    s1_with_ratio.select("VH_VV_Ratio")
    .reduce(ee.Reducer.percentile([10, 90]))
)

s1_max_ratio = s1_percentiles.select("VH_VV_Ratio_p90").rename("S1_Ratio_P90")
s1_min_ratio = s1_percentiles.select("VH_VV_Ratio_p10").rename("S1_Ratio_P10")
s1_variance  = (
    s1_with_ratio.select("VH_VV_Ratio")
    .reduce(ee.Reducer.stdDev())
    .rename("S1_Ratio_StdDev")
)

# Stack and mask
radar_features = (
    ee.Image.cat([s1_max_ratio, s1_min_ratio, s1_variance])
    .updateMask(valid_mask)
    .toFloat()
)

print("Radar features (all at 10m, masked):")
print("  S1_Ratio_P90   — peak canopy/volume (leaf-on)")
print("  S1_Ratio_P10   — peak ground/stem (leaf-off)")
print("  S1_Ratio_StdDev — structural plasticity")
print("\nradar_features ready.")

S1 observations loaded: 450 (window: 2014-04-01 → 2024-12-31)
Radar features (all at 10m, masked):
  S1_Ratio_P90   — peak canopy/volume (leaf-on)
  S1_Ratio_P10   — peak ground/stem (leaf-off)
  S1_Ratio_StdDev — structural plasticity

radar_features ready.


In [ ]:
# ============================================================
# CELL 7: Module 4 — Static Structural & Topographic Layers
#
# CHANGE FROM v1: citation corrected.
#   Lang et al. 2023 (Nature Ecology & Evolution) — the dataset
#   is for year 2020 but the paper is 2023. Previously cited
#   as "Lang 2020" which was wrong.
# ============================================================

# ── Lang et al. 2023 — Canopy Height (10m, 2020 snapshot) ─
canopy_h = (
    ee.Image("users/nlang/ETH_GlobalCanopyHeight_2020_10m_v1")
    .select("b1")
    .rename("Canopy_Height")
    .clip(roi)
    .unmask(0)          # Outside coverage → 0 (no canopy)
    .updateMask(valid_mask)
    .toFloat()
)
print("Lang et al. 2023 canopy height loaded (10m).")

# ── NASADEM — Elevation, Slope ─────────────────────────────
dem  = ee.Image("NASA/NASADEM_HGT/001").clip(roi)
elev = dem.select("elevation").rename("Elevation").updateMask(valid_mask).toFloat()
slp  = ee.Terrain.slope(elev).rename("Slope").updateMask(valid_mask).toFloat()
print("NASADEM elevation + slope loaded (30m, resampled to analysis scale later).")

# Stack
static_features = ee.Image.cat([canopy_h, elev, slp])

print("\nStatic features (3 bands):")
print("  Canopy_Height — Lang 2023, 10m, 2020 snapshot")
print("  Elevation     — NASADEM, 30m static")
print("  Slope         — derived from NASADEM")
print("\nstatic_features ready.")

Lang et al. 2023 canopy height loaded (10m).
NASADEM elevation + slope loaded (30m, resampled to analysis scale later).

Static features (3 bands):
  Canopy_Height — Lang 2023, 10m, 2020 snapshot
  Elevation     — NASADEM, 30m static
  Slope         — derived from NASADEM

static_features ready.


In [ ]:
# ============================================================
# CELL 8: Module 5 — Sentinel-2 Seasonal Composites
#
# These composites are ONLY used as SNIC input (Module 8).
# They are NOT used for phenology (HLS does that).
# SNIC asks: "where do spectral values change sharply
# across space?" — a spatial question needing just a few
# good cloud-free snapshots of the current landscape.
#
# WHY NOT HLS HERE: HLS is 30m. SNIC needs 10m to draw
# crisp stand boundaries. S2 gives 10m.
#
# SCL CLOUD MASK (the bug from v1 is fixed here):
#   v1 had mask_s2_clouds() defined but then the pipeline
#   re-built s2_base WITHOUT it. Fixed: we apply SCL masking
#   on every composite image below.
# ============================================================

def mask_s2_scl(img):
    """Per-pixel SCL cloud/shadow mask (the v1 bug fix)."""
    scl = img.select("SCL")
    # Keep: 4=Vegetation, 5=Bare soil, 6=Water (incl.), 7=Unclassified
    # Drop: 1=SatDefect, 2=DarkShadow, 3=CloudShadow, 8=CloudMed,
    #       9=CloudHigh, 10=Cirrus, 11=Snow
    keep = (scl.eq(4).Or(scl.eq(5)).Or(scl.eq(6))
               .Or(scl.eq(7)).Or(scl.eq(11).Not()))
    valid = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return img.updateMask(valid)

s2_year    = CONFIG["s2_year"]
seasons    = CONFIG["s2_seasons"]
cloud_pct  = CONFIG["cloud_pct_max"]

s2_composites = []
s2_band_names = []

for (season_name, start_mmdd, end_mmdd) in seasons:
    # Handle winter wrap-around (Nov–Jan crosses year boundary)
    start_yr = s2_year - 1 if season_name == "winter" else s2_year
    end_yr   = s2_year     if season_name == "winter" else s2_year

    start_date = f"{start_yr}-{start_mmdd}"
    end_date   = f"{end_yr}-{end_mmdd}"

    composite = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(roi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", cloud_pct))
        .map(mask_s2_scl)
        .select(["B2", "B3", "B4", "B8", "B11", "B12"])
        .median()
        .clip(roi)
    )

    # Rename bands to include season prefix for clarity
    for b in ["B2", "B3", "B4", "B8", "B11", "B12"]:
        s2_band_names.append(f"{season_name}_{b}")

    s2_composites.append(composite)
    print(f"  {season_name}: {start_date} → {end_date}")

# Stack all 4 composites into one multi-band image (4×6 = 24 bands)
s2_composite_stack = ee.Image.cat(s2_composites).rename(s2_band_names)
s2_composite_stack = s2_composite_stack.updateMask(valid_mask).toFloat()

print(f"\nS2 seasonal composite stack: {len(s2_band_names)} bands at 10m.")
print("This goes into SNIC only — NOT into the phenology module.")

  winter: 2022-11-01 → 2023-01-31
  pre_monsoon: 2023-03-01 → 2023-05-31
  monsoon: 2023-06-15 → 2023-09-15
  post_monsoon: 2023-09-16 → 2023-10-31

S2 seasonal composite stack: 24 bands at 10m.
This goes into SNIC only — NOT into the phenology module.


In [ ]:
# ============================================================
# CELL 9: Modules 6a + 7a — High-Res Stack & Pixel-Level Normalization
#
# WHAT'S NEW:
#   This is the SNIC input stack. It is DIFFERENT from the
#   clustering stack (built in Cell 11).
#
#   Inputs: S2 composites (10m) + S1 features (10m) + Lang
#   canopy height (10m).
#   HLS phenology (30m) is EXCLUDED here by design:
#   30m features would smear stand boundaries.
#
#   NORMALIZATION #1 (pixel-level):
#   Z-score across pixels in ROI. Ensures all features
#   contribute equally to SNIC's Euclidean distance metric.
#   (Different from Normalization #2 in Cell 11.)
#
# SNIC SEED SPACING:
#   Auto-scaled to target ~5000 stands regardless of ROI size.
#   For Sanjay Van (~4 km²): sqrt(4e6 / 5000) ≈ 28m → ~3 pixels at 10m.
# ============================================================

scale = CONFIG["analysis_scale"]

# ── Assemble the high-res feature stack (10m only) ────────
highres_stack_raw = ee.Image.cat([
    s2_composite_stack,   # 24 bands (4 seasons × 6 bands)
    radar_features,       # 3 bands (S1 ratio p10/p90/stdDev)
    canopy_h,             # 1 band  (Lang canopy height)
]).updateMask(valid_mask).toFloat()

hr_band_names = highres_stack_raw.bandNames()
print(f"High-res stack: {hr_band_names.size().getInfo()} bands at {scale}m")
print("(S2 composites + S1 ratio features + Lang canopy height)")

# ── Pixel-level Z-score normalization ─────────────────────
print("\nNormalization #1: pixel-level Z-score ...")

hr_stats = highres_stack_raw.reduceRegion(
    reducer   = ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
    geometry  = roi,
    scale     = scale,
    maxPixels = 1e9,
    bestEffort= True,
)

def normalize_band(b):
    b = ee.String(b)
    mean  = ee.Number(hr_stats.get(b.cat("_mean")))
    std   = ee.Number(hr_stats.get(b.cat("_stdDev"))).max(1e-6)
    return highres_stack_raw.select(b).subtract(mean).divide(std).rename(b)

highres_stack_norm = (
    ee.ImageCollection(hr_band_names.map(normalize_band))
    .toBands()
    .rename(hr_band_names)
    .updateMask(valid_mask)
    .toFloat()
)
print("High-res stack normalized (pixel Z-scores). Ready for SNIC.")

# ── Auto-scale SNIC seed spacing ──────────────────────────
# Target: ~5000 stands. seed_spacing in pixels.
# ROI area in m² / (target_stands) = m² per stand → sqrt = stand side in m
# Divide by scale to get spacing in pixels.

target_stands  = CONFIG["snic_target_stands"]
roi_area_m2    = roi.area(maxError=1).getInfo()  # exact area in m²
roi_area_km2   = roi_area_m2 / 1e6

stand_side_m   = math.sqrt(roi_area_m2 / target_stands)
seed_spacing_px = max(2, round(stand_side_m / scale))

print(f"\nROI area : {roi_area_km2:.2f} km²")
print(f"Target   : {target_stands} stands")
print(f"Stand side (target): {stand_side_m:.0f} m → seed spacing: {seed_spacing_px} px at {scale}m")

High-res stack: 28 bands at 10m
(S2 composites + S1 ratio features + Lang canopy height)

Normalization #1: pixel-level Z-score ...
High-res stack normalized (pixel Z-scores). Ready for SNIC.

ROI area : 13.04 km²
Target   : 5000 stands
Stand side (target): 51 m → seed spacing: 5 px at 10m


In [ ]:
# ============================================================
# CELL 10: Module 8 — SNIC Segmentation
#
# Runs on the NORMALIZED HIGH-RES STACK (from Cell 9).
# Phenology features are ABSENT here — they belong at the
# per-stand aggregation stage where they characterize
# stand-level seasonality, not at the pixel level.
#
# compactness=0.4: low → SNIC follows ecological boundaries,
# not geographic regularity. See §6 of design doc.
# ============================================================

compactness     = CONFIG["snic_compactness"]
connectivity    = CONFIG["snic_connectivity"]
neighborhood_sz = CONFIG["snic_neighborhood"]

print(f"Running SNIC with seed_spacing={seed_spacing_px}px, compactness={compactness} ...")

snic_seeds = ee.Algorithms.Image.Segmentation.seedGrid(seed_spacing_px)

snic = ee.Algorithms.Image.Segmentation.SNIC(
    image           = highres_stack_norm,
    compactness     = compactness,
    connectivity    = connectivity,
    neighborhoodSize= neighborhood_sz,
    seeds           = snic_seeds,
).reproject(crs="EPSG:4326", scale=scale)

# The '_mean' suffix bands are the stand-averaged values of
# every input feature for each SNIC superpixel.
object_means_hr = snic.select([".*_mean"])

print("SNIC superpixels generated.")
print("object_means_hr = per-pixel map of each stand's HR feature means.")

# ── Visualization ─────────────────────────────────────────
Map_snic = geemap.Map()
Map_snic.add_basemap("HYBRID")
Map_snic.centerObject(roi, 14)
Map_snic.addLayer(valid_mask.eq(0).selfMask(), {"palette": ["black"]},
                  "Excluded (urban/water)")
Map_snic.addLayer(snic.select("clusters").randomVisualizer().updateMask(valid_mask),
                  {}, "Module 8: SNIC Stand Boundaries")
Map_snic

Running SNIC with seed_spacing=5px, compactness=0.4 ...
SNIC superpixels generated.
object_means_hr = per-pixel map of each stand's HR feature means.


Map(center=[28.529999558822738, 77.17500000000798], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
# ============================================================
# CELL 11: Modules 6b + 9 + 10 — Full Feature Stack,
#          Per-Stand Aggregation, Stand-Level Normalization
#
# THIS IS THE BIG ARCHITECTURAL CHANGE FROM v1:
#
#   Module 6b: Build the FULL feature stack including the
#   HLS phenology features (6 bands). Phenology is at 30m
#   and is upsampled to 10m via BILINEAR resampling so it
#   sits on the same grid as SNIC stands. Upsampling does
#   NOT add spatial information — it is bookkeeping only.
#   (See §5 of design doc.)
#
#   Module 9: For each SNIC stand, compute the MEAN of every
#   feature in the full stack (zonal statistics). This
#   collapses from pixel-level to stand-level.
#
#   Module 10: NORMALIZATION #2 — stand-level Z-score.
#   This is a DIFFERENT normalization from Cell 9:
#     Cell 9: pixel distribution → unit-variance for SNIC.
#     Cell 11: stand distribution → unit-variance for K-means.
#   Averaging dampens variance, so these statistics differ.
# ============================================================

scale = CONFIG["analysis_scale"]

# ── Module 6b: Full feature stack ─────────────────────────
# Upsample 30m phenology features to 10m (bilinear).
# `.reproject()` forces evaluation at 10m before layering.
pheno_10m = pheno_features.reproject(
    crs   = "EPSG:4326",
    scale = scale,
).resample("bilinear")

# Upsample elevation/slope (30m → 10m) similarly
static_10m = static_features.reproject(
    crs   = "EPSG:4326",
    scale = scale,
).resample("bilinear")

full_stack_raw = ee.Image.cat([
    pheno_10m,      # Pheno_PC1..PC5 + Pheno_Trend  (6 bands, upsampled from 30m)
    radar_features, # S1_Ratio_P90/P10/StdDev        (3 bands, 10m native)
    canopy_h,       # Canopy_Height                  (1 band, 10m native)
    static_10m.select(["Elevation", "Slope"]),  # 2 bands, upsampled from 30m
]).updateMask(valid_mask).toFloat()

full_band_names = full_stack_raw.bandNames()
n_features = full_band_names.size().getInfo()
print(f"Full feature stack: {n_features} features at {scale}m")
print("Features:", full_band_names.getInfo())

# ── Module 9: Per-stand aggregation via SNIC cluster IDs ──
# For each SNIC stand, we need the mean of each full-stack
# feature. The cleanest GEE approach: add the cluster band
# to the full stack, then reduceConnectedComponents.

cluster_band   = snic.select("clusters")
full_stack_with_clusters = full_stack_raw.addBands(cluster_band)

# Reduce connected components: for each cluster, compute the
# mean of all full-stack bands.
stand_means = (
    full_stack_with_clusters
    .reduceConnectedComponents(
        reducer   = ee.Reducer.mean(),
        labelBand = "clusters",
    )
    .rename(full_band_names)
    .updateMask(valid_mask)
    .toFloat()
)
print("Per-stand aggregation complete (stand_means image).")

# ── Module 10: Stand-level Z-score normalization ──────────
print("\nNormalization #2: stand-level Z-score ...")

stand_stats = stand_means.reduceRegion(
    reducer   = ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
    geometry  = roi,
    scale     = scale,
    maxPixels = 1e9,
    bestEffort= True,
)

def normalize_stand_band(b):
    b     = ee.String(b)
    mean  = ee.Number(stand_stats.get(b.cat("_mean")))
    std   = ee.Number(stand_stats.get(b.cat("_stdDev"))).max(1e-6)
    return stand_means.select(b).subtract(mean).divide(std).rename(b)

normalized_stands = (
    ee.ImageCollection(full_band_names.map(normalize_stand_band))
    .toBands()
    .rename(full_band_names)
    .updateMask(valid_mask)
    .toFloat()
)

print("Stand-level normalization complete.")
print("normalized_stands ready for K-means clustering.")
print("\nNote: This normalization is statistically different from Cell 9's.")
print("Cell 9 = pixel distribution. Cell 11 = stand distribution.")
print("They are not redundant — each is fit for its stage.")

Full feature stack: 12 features at 10m
Features: ['Pheno_PC1', 'Pheno_PC2', 'Pheno_PC3', 'Pheno_PC4', 'Pheno_PC5', 'Pheno_Trend', 'S1_Ratio_P90', 'S1_Ratio_P10', 'S1_Ratio_StdDev', 'Canopy_Height', 'Elevation', 'Slope']
Per-stand aggregation complete (stand_means image).

Normalization #2: stand-level Z-score ...
Stand-level normalization complete.
normalized_stands ready for K-means clustering.

Note: This normalization is statistically different from Cell 9's.
Cell 9 = pixel distribution. Cell 11 = stand distribution.
They are not redundant — each is fit for its stage.


In [ ]:
# Get one sample per stand by also exporting the cluster ID
# then deduplicating in Python

stand_vectors = normalized_stands.addBands(
    snic.select("clusters").rename("stand_id")
)

sample_fc = stand_vectors.sample(
    region    = roi,
    scale     = scale,
    numPixels = 8000,   # oversample then deduplicate
    geometries= False,
)
sample_df = pd.DataFrame([f["properties"] for f in sample_fc.getInfo()["features"]])
sample_df = sample_df.drop_duplicates(subset=["stand_id"]).drop(columns=["stand_id"])
sample_df = sample_df.dropna()
X = sample_df.values
print(f"Unique stands sampled: {len(X)}")

In [ ]:
import datetime
timestamp = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
roi_slug  = CONFIG["roi_name"].lower().replace(",", "").replace(" ", "_")
scale     = CONFIG["analysis_scale"]

intermediate_id = CONFIG["export_asset_base"] + "normalized_stands_" + roi_slug

export_intermediate = ee.batch.Export.image.toAsset(
    image       = normalized_stands,
    description = "Export_NormalizedStands",
    assetId     = intermediate_id,
    region      = roi,
    scale       = scale,
    maxPixels   = 1e13,
)
export_intermediate.start()
print(f"Export started → {intermediate_id}")
print("Go to GEE Tasks tab and wait for it to complete before running Cell 12b.")

Export started → projects/replicating-paper/assets/normalized_stands_sanjay_van_delhi
Go to GEE Tasks tab and wait for it to complete before running Cell 12b.


In [ ]:
# ============================================================
# CELL 12: Module 11 — K-Means with Auto-K Selection
#
# CHANGE FROM v1: K=6 was hard-coded. Now we test K=2..15
# and select K using:
#   1. Silhouette score (higher is better, max ≈ 1)
#   2. Davies-Bouldin index (lower is better, min ≈ 0)
# Both computed in Python (sklearn) on a sample of stand vectors.
# The gap statistic is noted as a planned addition (Phase 3).
#
# NOTE: GEE's wekaKMeans is still used for the final map
# (it runs in GEE compute). The auto-K scoring samples data
# to Python only for the evaluation pass.
# ============================================================

k_range  = range(CONFIG["k_range"][0], CONFIG["k_range"][1] + 1)
n_sample = CONFIG["k_sample_pts"]

# Load the materialized asset — breaks the deep computation chain
norm_stands_asset = ee.Image(intermediate_id)

sample_fc = norm_stands_asset.sample(
    region    = roi,
    scale     = scale,
    numPixels = n_sample,
    geometries= False,
)
sample_df = pd.DataFrame([f["properties"] for f in sample_fc.getInfo()["features"]])
sample_df = sample_df.dropna()
X = sample_df.values
print(f"Sampled {len(X)} valid stand vectors ({X.shape[1]} features each).")


# ── Step 2: Score K = 2..15 ───────────────────────────────
print(f"Evaluating K from {k_range[0]} to {k_range[-1]} ...")

k_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)

    # Silhouette: closer to 1 is better
    sil = silhouette_score(X, labels, sample_size=min(500, len(X)), random_state=42)

    # Davies-Bouldin: lower is better
    db  = davies_bouldin_score(X, labels)

    k_scores.append({"K": k, "Silhouette": sil, "Davies_Bouldin": db})
    print(f"  K={k:2d}  Silhouette={sil:.3f}  DB={db:.3f}")

scores_df = pd.DataFrame(k_scores)

# ── Step 3: Select optimal K ──────────────────────────────
best_k_sil = scores_df.loc[scores_df["Silhouette"].idxmax(), "K"]
best_k_db  = scores_df.loc[scores_df["Davies_Bouldin"].idxmin(), "K"]

print(f"\nBest K by Silhouette score : K = {best_k_sil}")
print(f"Best K by Davies-Bouldin   : K = {best_k_db}")

if best_k_sil == best_k_db:
    OPTIMAL_K = int(best_k_sil)
    print(f"Consensus: OPTIMAL_K = {OPTIMAL_K}")
else:
    OPTIMAL_K = int(best_k_sil)  # Silhouette preferred; you can override
    print(f"Disagreement (differ by {abs(best_k_sil - best_k_db)}). "
          f"Defaulting to Silhouette → K = {OPTIMAL_K}. Review the plot below.")

# ── Step 4: Plot K selection curves ───────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(scores_df["K"], scores_df["Silhouette"], "b-o")
ax1.axvline(OPTIMAL_K, color="red", linestyle="--", label=f"Chosen K={OPTIMAL_K}")
ax1.set_xlabel("K"); ax1.set_ylabel("Silhouette Score")
ax1.set_title("Silhouette Score (↑ better)"); ax1.legend(); ax1.grid(True)

ax2.plot(scores_df["K"], scores_df["Davies_Bouldin"], "g-o")
ax2.axvline(OPTIMAL_K, color="red", linestyle="--", label=f"Chosen K={OPTIMAL_K}")
ax2.set_xlabel("K"); ax2.set_ylabel("Davies-Bouldin Index")
ax2.set_title("Davies-Bouldin Index (↓ better)"); ax2.legend(); ax2.grid(True)

plt.tight_layout()
plt.savefig("auto_k_selection.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved: auto_k_selection.png")

# ── Step 5: Final K-means clustering in GEE ───────────────
print(f"\nRunning final K-means (K={OPTIMAL_K}) on GEE ...")

training_data = norm_stands_asset.sample(
    region    = roi,
    scale     = scale,
    numPixels = 5000,
    geometries= True,
)
clusterer = ee.Clusterer.wekaKMeans(OPTIMAL_K, seed=42).train(training_data)

final_clusters = (
    norm_stands_asset
    .cluster(clusterer)
    .rename("Management_Unit")
    .updateMask(valid_mask)
)

cluster_palette = [
    "#e41a1c", "#377eb8", "#4daf4a", "#984ea3",
    "#ff7f00", "#ffff33", "#a65628", "#f781bf",
    "#999999", "#66c2a5", "#fc8d62", "#8da0cb",
    "#e78ac3", "#a6d854", "#ffd92f",
]

Map_kmeans = geemap.Map()
Map_kmeans.centerObject(roi, 14)
Map_kmeans.add_basemap("HYBRID")
Map_kmeans.addLayer(
    final_clusters,
    {"min": 0, "max": OPTIMAL_K - 1, "palette": cluster_palette[:OPTIMAL_K]},
    f"Module 11: K-means (K={OPTIMAL_K})",
)
print(f"K-means complete. {OPTIMAL_K} management units mapped.")
Map_kmeans

Sampling 1500 stand vectors for K evaluation ...


EEException: User memory limit exceeded.